# Triplet colocalization — HT/NALM vs HT/HB (per-cell Chung–Lu wedge Z)

Triplet analog of `ht_nalm_vs_ht_hb_analysis.ipynb`. Uses **per-cell** three-way wedge Z
for the pre-registered closed list (`triplet_candidate_list.md` → `triplet_candidate_list.json`,
171 distinct triplets), computed by `chunglu_triplets.py --triplet-list` (PNA graph is bipartite →
no triangles → every motif is an open wedge `W_{ABC}`).

Both systems share the **healthy T donor**; the **B target** differs:
* **HT/NALM** = `NALM-6 + healthy T` (B-ALL cell line)
* **HT/HB**   = `healthy B + healthy T` (primary healthy B)

Flow: load → build `coloc_triplet`/`coloc_doublet` obsm → HT/NALM vs HT/HB triplet differential →
Blina-vs-Mock per system (CD8, B) → module-grouped view → **doublet-vs-triplet Moran's I** benchmark
(à la `spatial_pca_benchmark.ipynb`).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen")
sys.path.insert(0, "/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from nalm_utils import compute_triplet_diff, split_triplet, display_name
from utils import build_spatial_pca_obsm
from metrics import distr_autocorrelation_in_latent

sc.set_figure_params(dpi=100, frameon=False)
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (5, 3), "figure.max_open_warning": 0})

BASE   = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data")
CACHE  = BASE / "cache"
OUT    = BASE / "results" / "chunglu"
ANNOTATED = CACHE / "adata_cytovi_annotated_compat.h5ad"

SYS_HT_NALM = "NALM-6 + healthy T"
SYS_HT_HB   = "healthy B + healthy T"
COLOR_NALM, COLOR_HB = "#9467bd", "#2ca02c"   # purple = ↑HT/NALM, green = ↑HT/HB

def tlabel(t):
    return "·".join(display_name(x) for x in split_triplet(t))

## Step 0 — Load adata + candidate-list metadata

In [ ]:
adata = sc.read_h5ad(ANNOTATED)
print("adata:", adata.shape, "| obsm:", list(adata.obsm.keys()))

cl = json.loads((OUT / "triplet_candidate_list.json").read_text())
trip_meta = pd.DataFrame(cl["meta"])
trip_meta["triplet"] = trip_meta["names"].apply(lambda x: "/".join(x))
trip_meta["module_short"] = trip_meta["module"].str.replace(r"Module (\d+).*", r"M\1", regex=True)
print("candidate triplets:", len(trip_meta))
display(trip_meta["module_short"].value_counts().sort_index())

## Step 1 — Build per-cell triplet & doublet colocalization obsm

`coloc_triplet` (`'A/B/C'` columns) from the new per-cell wedge parquets;
`coloc_doublet` (`'A/B'` columns) from the chunglu doublet parquets (for the apples-to-apples
Moran's I baseline). `*_asinh` variants stabilise heavy tails for the embedding/PCA.
Requires the LSF per-cell job (`run_chunglu_triplets_percell.lsf`) to have finished.

In [ ]:
def build_coloc_obsm(adata, df, value_col, prefix, marker_cols):
    df = df.copy()
    df["feat"] = df[marker_cols].astype(str).agg("/".join, axis=1)
    wide = (df.pivot_table(index="component", columns="feat", values=value_col, aggfunc="first")
              .reindex(adata.obs_names).fillna(0.0))
    base = wide.to_numpy(float)
    adata.obsm[prefix]            = pd.DataFrame(base, index=wide.index, columns=wide.columns)
    adata.obsm[prefix + "_asinh"] = pd.DataFrame(np.arcsinh(base), index=wide.index, columns=wide.columns)
    print(f"  obsm['{prefix}']: {adata.obsm[prefix].shape}  "
          f"(nonzero rate {np.mean(base != 0):.3f})")
    return wide.columns

# --- per-cell triplets (the new closed-list output) ---
pc_files = sorted(OUT.glob("*_triplets_percell.parquet"))
assert pc_files, "No *_triplets_percell.parquet yet — run run_chunglu_triplets_percell.lsf first."
trp = pd.concat([pd.read_parquet(p) for p in pc_files], ignore_index=True)
trp.to_parquet(OUT / "triplets_percell_all.parquet")
print(f"per-cell triplet rows: {len(trp):,}  ({trp['component'].nunique():,} cells)")
build_coloc_obsm(adata, trp, "triplet_z", "coloc_triplet", ["marker_1", "marker_2", "marker_3"])

# arcsinh(z/5): triplet analog of the doublet `spatial_asinh5` (tail-stabilised, matched scaling)
adata.obsm["coloc_triplet_asinh5"] = np.arcsinh(adata.obsm["coloc_triplet"] / 5)
print(f"  obsm['coloc_triplet_asinh5']: {adata.obsm['coloc_triplet_asinh5'].shape}  (arcsinh(z/5))")

# --- chunglu doublets (read minimal columns; large) for the Moran's-I baseline ---
BUILD_CHUNGLU_DOUBLET = True
if BUILD_CHUNGLU_DOUBLET:
    dbl = pd.concat([pd.read_parquet(p, columns=["component", "marker_1", "marker_2", "join_count_z"])
                     for p in sorted(OUT.glob("*_doublets.parquet"))], ignore_index=True)
    build_coloc_obsm(adata, dbl, "join_count_z", "coloc_doublet", ["marker_1", "marker_2"])
    del dbl

## Step 1b — Descriptive stats: distributions of the most variable triplet-Z

On the **asinh5-stabilised** z (`coloc_triplet_asinh5 = arcsinh(z/5)`) — raw triplet-Z is heavy-tailed, so
its histograms and variance ranking are dominated by a few outliers. Per cell type (CD8, B), pick the
**top-variance** features (variance over that cell type's cells in the two studied systems). For each, a
histogram of the per-cell z: first pooled over **all** cells of that type, then split by **experimental
setup** (`cell_system × condition`, time pooled): HT/NALM·Blina, HT/NALM·Mock, HT/HB·Blina, HT/HB·Mock.

In [ ]:
# Step 1b — most-variable triplet-Z distributions, all-cells vs per experimental setup
# Uses the asinh5-stabilised z: raw triplet-Z is heavy-tailed, so its histograms and variance
# ranking are dominated by a handful of outliers. (Differentials below stay on raw; see Step 7.)
DESC_KEY  = "coloc_triplet_asinh5"
SYSTEMS   = [SYS_HT_NALM, SYS_HT_HB]
SYS_SHORT = {SYS_HT_NALM: "HT/NALM", SYS_HT_HB: "HT/HB"}
TOP_N     = 9                                   # triplets shown per cell type
SETUP_ORDER = [f"{s} · {c}" for s in ("HT/NALM", "HT/HB") for c in ("Blinatumomab", "Mock")]
# 4 distinct, colorblind-safe hues (Okabe–Ito): blue, orange, green, purple
SETUP_PAL   = dict(zip(SETUP_ORDER, ["#0072b2", "#e69f00", "#009e73", "#cc79a7"]))

def _ct_slice(cell_type, obsm_key="coloc_triplet"):
    o = adata.obs
    m = ((o["cell_type_annot"] == cell_type) & (o["cell_system"].isin(SYSTEMS))).values
    sub   = adata.obsm[obsm_key].loc[m]
    setup = (o.loc[m, "cell_system"].map(SYS_SHORT).astype(str)
             + " · " + o.loc[m, "condition"].astype(str)).values
    return sub, setup

def triplet_hist_grid(sub, top, title, setup=None, ncols=3):
    nrows = (len(top) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.7 * nrows), squeeze=False)
    for i, (ax, t) in enumerate(zip(axes.flat, top)):
        if setup is None:
            sns.histplot(sub[t].to_numpy(float), bins=40, stat="density", kde=True,
                         color="#4c72b0", ax=ax)
        else:
            d = pd.DataFrame({"z": sub[t].to_numpy(float), "setup": setup})
            sns.histplot(d, x="z", hue="setup", hue_order=SETUP_ORDER, palette=SETUP_PAL,
                         bins=40, stat="density", common_norm=False, element="step",
                         fill=False, kde=True, lw=1.6, legend=(i == 0), ax=ax)
            if i == 0:
                sns.move_legend(ax, "upper left", fontsize=6, title=None, frameon=False)
        ax.set_title(tlabel(t), fontsize=8); ax.set_xlabel("triplet Z (asinh5)"); ax.set_ylabel("")
    for ax in axes.flat[len(top):]:
        ax.axis("off")
    fig.suptitle(title, y=1.0); plt.tight_layout(); plt.show()

for ct in ["CD8", "B"]:
    sub, setup = _ct_slice(ct, DESC_KEY)
    top = sub.var().nlargest(TOP_N).index.tolist()
    desc = (sub[top].agg(["mean", "std", "var"]).T
            .assign(triplet=lambda d: [tlabel(t) for t in d.index])
            .sort_values("var", ascending=False)[["triplet", "mean", "std", "var"]].round(3))
    print(f"\n=== {ct}: {len(sub):,} cells (HT/NALM ∪ HT/HB) — top {TOP_N} most variable triplet-Z [asinh5] ===")
    display(desc.reset_index(drop=True))
    triplet_hist_grid(sub, top, f"{ct} — top {TOP_N} most variable triplet-Z [asinh5] (all cells)")
    triplet_hist_grid(sub, top, f"{ct} — top {TOP_N} most variable triplet-Z [asinh5] (by setup)", setup=setup)

## Step 2 — HT/NALM vs HT/HB triplet differential (CD8, B)

Per cell type, Mann–Whitney + BH-FDR on the per-cell triplet Z (`compute_triplet_diff`);
`mean_diff = HT/NALM − HT/HB`. Cells are the unit; blocked within cell type as in the doublet notebook.

In [ ]:
def system_diff(adata, cell_type, obsm_key="coloc_triplet"):
    sp = adata.obsm[obsm_key]
    m_n = ((adata.obs["cell_system"] == SYS_HT_NALM) & (adata.obs["cell_type_annot"] == cell_type)).values
    m_h = ((adata.obs["cell_system"] == SYS_HT_HB)   & (adata.obs["cell_type_annot"] == cell_type)).values
    res = compute_triplet_diff(sp.loc[m_n], sp.loc[m_h])     # mean_a=NALM, mean_b=HB
    res["n_nalm"], res["n_hb"] = int(m_n.sum()), int(m_h.sum())
    return res

def triplet_volcano(res, title, ax):
    sig = res["padj"] < 0.05
    ax.scatter(res.loc[~sig, "mean_diff"], -np.log10(res.loc[~sig, "padj"]),
               s=10, color="#bbbbbb", alpha=.6)
    ax.scatter(res.loc[sig, "mean_diff"], -np.log10(res.loc[sig, "padj"]),
               s=14, c=np.where(res.loc[sig, "mean_diff"] > 0, COLOR_NALM, COLOR_HB))
    ax.axhline(-np.log10(0.05), ls="--", lw=.7, color="k")
    ax.axvline(0, ls="--", lw=.7, color="k")
    for _, r in res[sig].reindex(res[sig]["mean_diff"].abs().sort_values(ascending=False).index).head(8).iterrows():
        ax.annotate(tlabel(r["triplet"]), (r["mean_diff"], -np.log10(r["padj"])),
                    fontsize=6, xytext=(2, 2), textcoords="offset points")
    ax.set_xlabel("Δ mean triplet-Z  (HT/NALM − HT/HB)"); ax.set_ylabel("−log10 FDR")
    ax.set_title(title)

res_cd8 = system_diff(adata, "CD8")
res_b   = system_diff(adata, "B")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
triplet_volcano(res_cd8, f"CD8  ({res_cd8['n_nalm'].iloc[0]} NALM / {res_cd8['n_hb'].iloc[0]} HB)", axes[0])
triplet_volcano(res_b,   f"B  ({res_b['n_nalm'].iloc[0]} NALM / {res_b['n_hb'].iloc[0]} HB)",   axes[1])
plt.tight_layout(); plt.show()

for nm, res in [("CD8", res_cd8), ("B", res_b)]:
    print(f"\n=== {nm}: {(res.padj < 0.05).sum()} triplets FDR<0.05 (of {len(res)}) ===")
    top = res.reindex(res["mean_diff"].abs().sort_values(ascending=False).index).head(10).copy()
    top["triplet"] = top["triplet"].map(tlabel)
    display(top[["triplet", "mean_a", "mean_b", "mean_diff", "padj"]]
            .rename(columns={"mean_a": "HT/NALM", "mean_b": "HT/HB"}).round(3))

## Step 3 — Blina vs Mock per system (CD8, B)

Triplet analog of the doublet `plot_condition_comparison` / MA panels. For each system, per-cell
Mann–Whitney Blina vs Mock on the triplet Z, top-N movers + MA (LFC vs mean signal).

In [ ]:
def cond_diff(adata, cell_type, system, obsm_key="coloc_triplet", time_val="6h"):
    sp = adata.obsm[obsm_key]; o = adata.obs
    base = (o["cell_type_annot"] == cell_type) & (o["cell_system"] == system) & (o["time"] == time_val)
    m_a = (base & (o["condition"] == "Blinatumomab")).values   # a = Blina
    m_b = (base & (o["condition"] == "Mock")).values           # b = Mock
    res = compute_triplet_diff(sp.loc[m_a], sp.loc[m_b])        # mean_diff = Blina − Mock
    res["mean_signal"] = (res["mean_a"] + res["mean_b"]) / 2
    res["n_blina"], res["n_mock"] = int(m_a.sum()), int(m_b.sum())
    return res

def top_bars(res, top_n, title):
    up = res.nlargest(top_n, "mean_diff").sort_values("mean_diff")
    dn = res.nsmallest(top_n, "mean_diff").sort_values("mean_diff")
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, sub, col, lab in [(axes[0], dn, "#1f77b4", "higher in Mock"),
                              (axes[1], up, "#d62728", "higher in Blina")]:
        ax.barh([tlabel(t) for t in sub["triplet"]], sub["mean_diff"], color=col, alpha=.85)
        ax.axvline(0, color="k", lw=.7, ls="--"); ax.tick_params(axis="y", labelsize=7)
        ax.set_xlabel("Δ mean triplet-Z (Blina − Mock)"); ax.set_title(f"{title} — top {top_n} {lab}")
    plt.tight_layout(); plt.show()

def ma_panel(results, titles, suptitle):
    fig, axes = plt.subplots(1, len(results), figsize=(7.5 * len(results), 5), squeeze=False)
    for ax, res, t in zip(axes[0], results, titles):
        sig = res["padj"] < 0.05
        ax.scatter(res.loc[~sig, "mean_signal"], res.loc[~sig, "mean_diff"], s=10, color="lightgrey", alpha=.6)
        ax.scatter(res.loc[sig & (res.mean_diff < 0), "mean_signal"], res.loc[sig & (res.mean_diff < 0), "mean_diff"],
                   s=16, color="#1f77b4", label="↑Mock")
        ax.scatter(res.loc[sig & (res.mean_diff > 0), "mean_signal"], res.loc[sig & (res.mean_diff > 0), "mean_diff"],
                   s=16, color="#d62728", label="↑Blina")
        ax.axhline(0, color="k", lw=.7, ls="--")
        for _, r in res[sig].reindex(res[sig]["mean_diff"].abs().sort_values(ascending=False).index).head(8).iterrows():
            ax.annotate(tlabel(r["triplet"]), (r["mean_signal"], r["mean_diff"]), fontsize=6,
                        xytext=(2, 2), textcoords="offset points")
        ax.set_xlabel("mean triplet-Z (Blina ∪ Mock)"); ax.set_ylabel("Δ (Blina − Mock)")
        ax.set_title(t); ax.legend(fontsize=8, frameon=False)
    fig.suptitle(suptitle, y=1.02); plt.tight_layout(); plt.show()

for ct in ["CD8", "B"]:
    rn = cond_diff(adata, ct, SYS_HT_NALM); rh = cond_diff(adata, ct, SYS_HT_HB)
    print(f"\n##### {ct}: HT/NALM (Blina {rn['n_blina'].iloc[0]}/Mock {rn['n_mock'].iloc[0]}), "
          f"HT/HB (Blina {rh['n_blina'].iloc[0]}/Mock {rh['n_mock'].iloc[0]}) #####")
    top_bars(rn, 12, f"{ct} · HT/NALM · 6h")
    top_bars(rh, 12, f"{ct} · HT/HB · 6h")
    ma_panel([rn, rh], [f"{ct} · HT/NALM", f"{ct} · HT/HB"],
             f"{ct} — Blina vs Mock triplet MA (6h)")

### Spatial diff scatter — Blina−Mock, HT/HB (x) vs HT/NALM (y)

System-vs-system contrast of the Blina-induced triplet shift (mirrors the doublet diff scatter).
Points above y=x shifted more in HT/NALM; below, more in HT/HB.

In [ ]:
def diff_scatter(adata, cell_type, obsm_key="coloc_triplet", time_val="6h", top_k=10):
    def shift(system):
        r = cond_diff(adata, cell_type, system, obsm_key, time_val)
        return r.set_index("triplet")["mean_diff"]
    a = shift(SYS_HT_HB); b = shift(SYS_HT_NALM)
    df = pd.DataFrame({"a": a, "b": b}).dropna()
    df["off"] = (df["b"] - df["a"]) / np.sqrt(2)
    top = df["off"].abs().nlargest(top_k).index
    fig, ax = plt.subplots(figsize=(7, 7))
    lim = max(df[["a", "b"]].abs().max().max() * 1.15, 0.05)
    ax.plot([-lim, lim], [-lim, lim], ls=":", color="grey")
    ax.axhline(0, color="grey", lw=.6, ls="--"); ax.axvline(0, color="grey", lw=.6, ls="--")
    rest = df.index.difference(top)
    ax.scatter(df.loc[rest, "a"], df.loc[rest, "b"], s=18, color="#bbbbbb", alpha=.55)
    ax.scatter(df.loc[top, "a"], df.loc[top, "b"], s=34,
               c=np.where(df.loc[top, "off"] > 0, COLOR_NALM, COLOR_HB))
    for t in top:
        ax.annotate(tlabel(t), (df.loc[t, "a"], df.loc[t, "b"]), fontsize=6,
                    xytext=(2, 2), textcoords="offset points")
    ax.set_xlabel("Δ(Blina−Mock) — HT/HB"); ax.set_ylabel("Δ(Blina−Mock) — HT/NALM")
    ax.set_title(f"{cell_type} — triplet Blina-shift, HT/NALM vs HT/HB (6h)")
    plt.tight_layout(); plt.show()

    # triplets farthest from the diagonal (hard to read off the figure)
    tbl = (df.loc[top].reindex(df.loc[top, "off"].abs().sort_values(ascending=False).index)
             .assign(triplet=lambda d: [tlabel(t) for t in d.index])
             .rename(columns={"a": "Δ HT/HB", "b": "Δ HT/NALM", "off": "off_diag"})
             [["triplet", "Δ HT/HB", "Δ HT/NALM", "off_diag"]].round(3).reset_index(drop=True))
    print(f"{cell_type} — top {top_k} triplets farthest from diagonal "
          f"(off_diag>0 ⇒ larger Blina shift in HT/NALM [purple], <0 ⇒ in HT/HB [green]):")
    display(tbl)
    return tbl

for ct in ["CD8", "B"]:
    diff_scatter(adata, ct)

## Step 4 — Module-grouped view (the triplet analog of the synapse suites)

The candidate list is organised into 19 synapse modules; group the HT/NALM−HT/HB differential by
module (replaces the pairwise heatmap/network, which is doublet-only).

In [ ]:
def module_summary(res, cell_type):
    m = res.merge(trip_meta[["triplet", "module_short", "hub", "sign"]], on="triplet", how="left")
    g = (m.groupby("module_short")
           .agg(n=("triplet", "size"), mean_dZ=("mean_diff", "mean"),
                n_sig=("padj", lambda s: int((s < 0.05).sum())))
           .sort_values("mean_dZ"))
    fig, ax = plt.subplots(figsize=(7, max(3, 0.32 * len(g))))
    ax.barh(g.index, g["mean_dZ"], color=np.where(g["mean_dZ"] > 0, COLOR_NALM, COLOR_HB), alpha=.85)
    ax.axvline(0, color="k", lw=.7, ls="--")
    for i, (_, r) in enumerate(g.iterrows()):
        ax.text(r["mean_dZ"], i, f"  {r['n_sig']}/{r['n']}", va="center", fontsize=7)
    ax.set_xlabel("mean Δ triplet-Z (HT/NALM − HT/HB)")
    ax.set_title(f"{cell_type} — per-module triplet shift  (n_sig/n labelled)")
    plt.tight_layout(); plt.show()
    return g

_ = module_summary(res_cd8, "CD8")
_ = module_summary(res_b, "B")

## Step 5 — Doublet-vs-triplet Moran's I on the main embedding

Mirrors `spatial_pca_benchmark.ipynb` §7. Build a PCA "main embedding" per colocalization modality,
then compute Moran's I of each abundance marker under that embedding's kNN graph. Higher Moran's I =
the modality's embedding organises cells along smoother feature axes.

In [ ]:
adata_ac = adata.copy()
EMB_SOURCES = [("coloc_triplet_asinh", "triplet")]
if "coloc_triplet_asinh5" in adata_ac.obsm:
    EMB_SOURCES.append(("coloc_triplet_asinh5", "triplet_asinh5"))   # arcsinh(z/5), matched to spatial_asinh5
if "coloc_doublet_asinh" in adata_ac.obsm:
    EMB_SOURCES.append(("coloc_doublet_asinh", "chunglu_doublet"))
if "spatial_asinh5" in adata_ac.obsm:
    EMB_SOURCES.append(("spatial_asinh5", "proximity_doublet"))   # pixelator pg.proximity join-count z

latent_keys, names = [], []
for src, nm in EMB_SOURCES:
    k = build_spatial_pca_obsm(adata_ac, source_key=src, n_components=20, standardize="center")
    latent_keys.append(k); names.append(nm)
print("embedding obsm keys:", latent_keys)

ac = distr_autocorrelation_in_latent(adata_ac, latent_keys=latent_keys, names=names,
                                     rep_key="arcsinh", pca_kwargs={"n_comps": 15})
print("Moran's I rows:", ac.shape)
ac.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=ac, x="morans", hue="latent", kde=True, stat="density",
             common_norm=False, alpha=.4, ax=ax)
ax.set_title("Abundance Moran's I under each colocalization embedding")
ax.set_xlabel("Moran's I"); plt.tight_layout(); plt.show()

summary = (ac.groupby("latent")["morans"].agg(["mean", "median", "std"]).round(4))
display(summary)

# paired scatter: triplet vs each doublet embedding (Moran's I per abundance feature)
doublet_names = [n for n in names if n != "triplet"]
if doublet_names:
    fig, axes = plt.subplots(1, len(doublet_names), figsize=(6 * len(doublet_names), 5.5), squeeze=False)
    yt = ac[ac["latent"] == "triplet"]["morans"]
    for ax, dn in zip(axes[0], doublet_names):
        m = ac[ac["latent"] == dn]["morans"]
        common = m.index.intersection(yt.index)
        x, y = m.loc[common].values, yt.loc[common].values
        ax.scatter(x, y, s=14, alpha=.5)
        lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=1, alpha=.5); ax.set_aspect("equal")
        ax.set_xlabel(f"Moran's I — {dn}"); ax.set_ylabel("Moran's I — triplet")
        ax.set_title(f"triplet vs {dn}  ({int((y > x).sum())}/{len(x)} above diag)")
    plt.tight_layout(); plt.show()

## Step 5b — Coloc-feature Moran's I on an *abundance* embedding

Inverse of Step 5. Here the kNN graph is built on the **abundance** latent (`X_CytoVI`) and we measure
the Moran's I of each **coloc feature** (triplet vs chunglu-doublet vs pixelator-proximity-doublet) on
that graph. `chunglu_doublet` and `proximity_doublet` (`spatial_raw`) are **two join-count z-scores on
the same bipartite proximity graph with different nulls** (analytic Chung–Lu fixed-degree vs pixelator
`pg.proximity()`; see Step 5c) — *not* Hotspot. High Moran's I = the triplet/doublet statistic varies
smoothly across the abundance-defined manifold (it tracks cell state); low = spatial structure largely
orthogonal to abundance.

In [ ]:
# Step 5b — Moran's I of the coloc features on an *abundance* embedding
# (inverse of Step 5: graph = abundance, features = the spatial coloc statistics)
ABUND_EMB = "X_CytoVI"          # abundance latent; kNN graph built on this (alt: "X_pca")
adata_ab = adata.copy()

# rep_keys are the RAW per-cell z features (match raw coloc_triplet / coloc_doublet);
# spatial_raw = pixelator pg.proximity join-count z (NOT hotspot — see Step 5c).
rep_sets = [("coloc_triplet", "triplet")]
if "coloc_doublet" in adata_ab.obsm:
    rep_sets.append(("coloc_doublet", "chunglu_doublet"))
if "spatial_raw" in adata_ab.obsm:
    rep_sets.append(("spatial_raw", "proximity_doublet"))

ac_ab = []
for rep_key, feat_name in rep_sets:
    r = distr_autocorrelation_in_latent(
        adata_ab, latent_keys=[ABUND_EMB], names=[feat_name],
        rep_key=rep_key, pca_kwargs={"n_comps": 15})
    ac_ab.append(r)
# feature names repeat across sets -> flatten to a column (seaborn KDE can't reindex dup labels)
ac_ab = (pd.concat(ac_ab).rename(columns={"latent": "feature_set"})
           .rename_axis("feature").reset_index())
print("Moran's I rows:", ac_ab.shape)

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=ac_ab, x="morans", hue="feature_set", kde=True, stat="density",
             common_norm=False, alpha=.4, ax=ax)
ax.set_title(f"Coloc-feature Moran's I on the abundance ({ABUND_EMB}) embedding")
ax.set_xlabel("Moran's I"); plt.tight_layout(); plt.show()

display(ac_ab.groupby("feature_set")["morans"].agg(["mean", "median", "std", "count"]).round(4))

# triplet features that most / least track the abundance manifold
tri = ac_ab[ac_ab["feature_set"] == "triplet"].sort_values("morans", ascending=False)
show = pd.concat([tri.head(10), tri.tail(10)]).copy()
show["triplet"] = [tlabel(t) for t in show["feature"]]
print("triplet features — highest & lowest Moran's I on the abundance embedding:")
display(show[["triplet", "morans", "gearys"]].round(3).reset_index(drop=True))

## Step 5c — pixelator `pg.proximity()` vs chunglu: same statistic, different null

Sanity check on the two doublet references used above. **Both are per-cell join-count z-scores on the
same bipartite PNA proximity graph** — they differ only in the null:
* `proximity_doublet` (`spatial_raw`) — pixelator **`pg.proximity()`** (PNA `--compute-proximity`), the
  pipeline's built-in colocalization z.
* `chunglu_doublet` (`coloc_doublet`) — the analytic **bipartite Chung–Lu fixed-degree** null
  (`chunglu_triplets.py`), the per-side degree-corrected estimator this project introduced.

Neither is Hotspot (Hotspot's `coloc_hs_z` is a separate obsm, absent here). Expect strong but imperfect
agreement (~0.7 Pearson, with a systematic centering/scale offset) — the discrepancy *is* the null-model
difference, which is exactly what the chunglu correction targets.

In [ ]:
# Step 5c — pixelator pg.proximity vs chunglu doublet: same statistic, different null
# Both are per-cell join-count z on the bipartite PNA graph. proximity (spatial_raw) uses
# pixelator's pg.proximity() null; chunglu (coloc_doublet) uses the analytic bipartite
# Chung–Lu fixed-degree null. Align on canonical (unordered) pair key and correlate.
def _canon(p):
    x, y = p.split("/"); return f"{x}/{y}" if x <= y else f"{y}/{x}"

prox = adata.obsm["spatial_raw"]      # pixelator pg.proximity join-count z
chun = adata.obsm["coloc_doublet"]    # chunglu fixed-degree join-count z
prox_c = prox.rename(columns={c: _canon(c) for c in prox.columns}).loc[:, lambda d: ~d.columns.duplicated()]
chun_c = chun.rename(columns={c: _canon(c) for c in chun.columns}).loc[:, lambda d: ~d.columns.duplicated()]
common = sorted(set(prox_c.columns) & set(chun_c.columns))
P, C = prox_c[common], chun_c[common]

per_pair = P.corrwith(C)              # Pearson per pair across cells (vectorised)
rng = np.random.default_rng(0)        # subsample pairs to bound the pooled scatter memory
samp = list(rng.choice(common, size=min(1500, len(common)), replace=False))
pv, cv = P[samp].to_numpy().ravel(), C[samp].to_numpy().ravel()
ok = np.isfinite(pv) & np.isfinite(cv)
pooled = np.corrcoef(cv[ok], pv[ok])[0, 1]
print(f"{len(common):,} common pairs | pooled Pearson={pooled:.3f} | "
      f"per-pair median r={per_pair.median():.3f}  (IQR {per_pair.quantile(.25):.2f}–{per_pair.quantile(.75):.2f})")
print(f"means -> proximity {P.values.mean():.2f} | chunglu {C.values.mean():.2f}   "
      f"(std {P.values.std():.2f} vs {C.values.std():.2f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(per_pair.dropna(), bins=40, color="#4878CF")
axes[0].axvline(per_pair.median(), color="k", ls="--", lw=.8)
axes[0].set_xlabel("per-pair Pearson r (across cells)"); axes[0].set_ylabel("# pairs")
axes[0].set_title(f"proximity vs chunglu agreement (median r={per_pair.median():.2f})")
lim = 10
hb = axes[1].hexbin(cv[ok], pv[ok], gridsize=60, bins="log", cmap="viridis", extent=(-lim, lim, -lim, lim))
axes[1].plot([-lim, lim], [-lim, lim], "r--", lw=.8); axes[1].set_xlim(-lim, lim); axes[1].set_ylim(-lim, lim)
axes[1].set_xlabel("chunglu join-count z"); axes[1].set_ylabel("pixelator pg.proximity z")
axes[1].set_title(f"same signal, different null (pooled r={pooled:.2f})")
fig.colorbar(hb, ax=axes[1], label="log10 count")
plt.tight_layout(); plt.show()

## Step 6 — UMAP coloured by the most variable / important triplets

Project the per-cell **asinh5-stabilised** triplet Z (`coloc_triplet_asinh5 = arcsinh(z/5)`) onto the
abundance UMAP (`X_umap`). Two panels of small multiples: (i) the 12 highest-variance triplet features,
(ii) the strongest HT/NALM−HT/HB differential triplets (CD8 ∪ B, FDR<0.05). Diverging colour centred at 0;
clipped at the 99th percentile of |Z|.

In [ ]:
# Step 6 — UMAP coloured by the most variable / differential triplets (asinh5-stabilised z)
UMAP_KEY = "X_umap"                 # abundance UMAP already in adata (alt: "X_umap_cytovi")
emb    = adata.obsm[UMAP_KEY]
sp_tri = adata.obsm["coloc_triplet_asinh5"]   # arcsinh(z/5): readable diverging colour scale

def umap_triplets(triplets, title, ncols=4, sp=None):
    sp = sp_tri if sp is None else sp
    seen = list(dict.fromkeys(t for t in triplets if t in sp.columns))
    n = len(seen); nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 3 * nrows), squeeze=False)
    for ax, t in zip(axes.flat, seen):
        c = sp[t].to_numpy(float)
        v = np.quantile(np.abs(c), 0.99) or 1.0
        s = ax.scatter(emb[:, 0], emb[:, 1], c=c, cmap="RdBu_r", vmin=-v, vmax=v, s=3, lw=0)
        ax.set_title(tlabel(t), fontsize=7); ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(s, ax=ax, fraction=.046, pad=.04)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle(title, y=1.0); plt.tight_layout(); plt.show()

# reference panels: cell type & system on the same UMAP
sc.pl.umap(adata, color=["cell_type_annot", "cell_system"], ncols=2, wspace=.35)

# (i) most variable triplet Z features
most_var = sp_tri.var().nlargest(12).index.tolist()
umap_triplets(most_var, "UMAP — most variable triplet Z [asinh5]")

# (ii) strongest HT/NALM−HT/HB differential triplets (CD8 ∪ B, FDR<0.05)
def top_diff(res, k=6):
    sig = res[res["padj"] < 0.05]
    order = sig["mean_diff"].abs().sort_values(ascending=False).index
    return sig.loc[order, "triplet"].head(k).tolist()
important = top_diff(res_cd8) + top_diff(res_b)
umap_triplets(important, "UMAP — top HT/NALM vs HT/HB differential triplets (CD8 ∪ B) [asinh5]")

## Step 7 — Differential pipeline on the arcsinh(z/5)-stabilised triplet Z

`coloc_triplet_asinh5 = arcsinh(coloc_triplet / 5)` — the triplet analog of the doublet `spatial_asinh5`
(tail-stabilised, matched scaling). Re-runs **Steps 2 → 4** on this transformed Z via one
`run_triplet_analysis(obsm_key, tag)` reusing the same functions. (Step 1b's descriptive distributions and
Step 6's UMAPs already use asinh5 in place.) Mann–Whitney + BH-FDR is rank-invariant under the monotone
transform, so **p-values/FDR are unchanged**; only the **effect-size scale** (Δ mean Z) changes. The asinh5
triplet embedding is also added to the Step 5 Moran's comparison above. Raw-Z differentials (Steps 2–4)
remain above for side-by-side reading.

In [ ]:
# Step 7 — re-run the differential pipeline on coloc_triplet_asinh5 (arcsinh(z/5))
# (descriptive distributions use asinh5 in Step 1b; UMAPs use asinh5 in Step 6; here the effect-size analyses.)
def run_triplet_analysis(obsm_key, tag):
    # --- Step 2: HT/NALM vs HT/HB differential (FDR identical to raw; effect sizes rescaled) ---
    rc = system_diff(adata, "CD8", obsm_key); rb = system_diff(adata, "B", obsm_key)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    triplet_volcano(rc, f"CD8  ({rc['n_nalm'].iloc[0]} NALM / {rc['n_hb'].iloc[0]} HB) [{tag}]", axes[0])
    triplet_volcano(rb, f"B  ({rb['n_nalm'].iloc[0]} NALM / {rb['n_hb'].iloc[0]} HB) [{tag}]",   axes[1])
    plt.tight_layout(); plt.show()
    for nm, res in [("CD8", rc), ("B", rb)]:
        print(f"\n=== {nm} [{tag}]: {(res.padj < 0.05).sum()} triplets FDR<0.05 (of {len(res)}) ===")
        t = res.reindex(res["mean_diff"].abs().sort_values(ascending=False).index).head(10).copy()
        t["triplet"] = t["triplet"].map(tlabel)
        display(t[["triplet", "mean_a", "mean_b", "mean_diff", "padj"]]
                .rename(columns={"mean_a": "HT/NALM", "mean_b": "HT/HB"}).round(3))

    # --- Step 3: Blina vs Mock per system ---
    for ct in ["CD8", "B"]:
        rn = cond_diff(adata, ct, SYS_HT_NALM, obsm_key); rh = cond_diff(adata, ct, SYS_HT_HB, obsm_key)
        print(f"\n##### {ct} [{tag}]: HT/NALM (Blina {rn['n_blina'].iloc[0]}/Mock {rn['n_mock'].iloc[0]}), "
              f"HT/HB (Blina {rh['n_blina'].iloc[0]}/Mock {rh['n_mock'].iloc[0]}) #####")
        top_bars(rn, 12, f"{ct} · HT/NALM · 6h [{tag}]")
        top_bars(rh, 12, f"{ct} · HT/HB · 6h [{tag}]")
        ma_panel([rn, rh], [f"{ct} · HT/NALM", f"{ct} · HT/HB"], f"{ct} — Blina vs Mock triplet MA (6h) [{tag}]")

    # --- Step 3.5: system-vs-system Blina-shift scatter ---
    for ct in ["CD8", "B"]:
        diff_scatter(adata, ct, obsm_key)

    # --- Step 4: module-grouped view ---
    module_summary(rc, "CD8"); module_summary(rb, "B")
    return rc, rb

res_cd8_a5, res_b_a5 = run_triplet_analysis("coloc_triplet_asinh5", "asinh5")

## Methods note

- **Per-cell triplet Z**: bipartite Chung–Lu / fixed-degree wedge null (`chunglu_triplets.py`,
  `triplet_list_table`). Observed `W_{ABC}` = sum over the 3 possible hub centers; `EW`/`Var` from the
  falling-factorial closed form (validated max|Δ|=0 vs the dense K³ path). No triangles in a bipartite graph.
- **Closed list**: 171 distinct triplets from `triplet_candidate_list.md` (188 module rows − 1 self-pair
  − CD63-absent #138, deduped). §F contrasts (#189–208) and self-pair polarization anchors (#209–218) excluded.
- **obsm**: `coloc_triplet` (`'A/B/C'`, raw `triplet_z`) + `_asinh` (plain arcsinh) + `_asinh5`
  (`arcsinh(z/5)`, matched to the doublet `spatial_asinh5` scaling — Step 7 / Moran's); `coloc_doublet`
  from chunglu `join_count_z`; cells joined by `component` (= adata index).
- **Doublet references** (both join-count z on the same bipartite PNA graph, *different nulls*):
  `coloc_doublet` = analytic bipartite **Chung–Lu fixed-degree** join-count z (`chunglu_triplets.py`);
  `proximity_doublet` (`spatial_raw`) = pixelator **`pg.proximity()`** join-count z (PNA `--compute-proximity`).
  Same statistic family → ~0.73 Pearson, different centering/scale (Step 5c). `spatial_asinh5 = arcsinh(spatial_raw/5)`.
  There is **no Hotspot doublet here** — Hotspot's `coloc_hs_z` (`pxl_utils.compute_hotspot_pol_and_coloc`) is
  a separate obsm and is absent from this h5ad.
- **Differentials**: cell-level Mann–Whitney + BH-FDR (`compute_triplet_diff`); HT/NALM vs HT/HB and
  Blina vs Mock per system, CD8 and B, 6h. Cells are the replication unit. The transform is monotone, so
  Step 7 (asinh5) shares the raw FDR; only effect sizes (Δ mean Z) and distributions change.
- **Moran's I**: `build_spatial_pca_obsm` (20 PCs) per modality → `distr_autocorrelation_in_latent`
  (kNN on the embedding, Moran's I of each abundance marker). Higher = smoother organisation.